In [ ]:
storage_account = "stretailbidevgk01"
container = "retail-data"

storage_key = dbutils.secrets.get(
    scope="kv-retail-bi-dev",
    key="adls-storage-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("ADLS authentication configured successfully")

In [ ]:
orders_silver_path = "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/silver/wwi/orders/"
orderlines_silver_path = "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/silver/wwi/orderlines/"
customers_silver_path = "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/silver/wwi/customers/"
weather_silver_path = "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/silver/weather/"
holiday_silver_path = "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/silver/holiday/"

df_orders = spark.read.format("delta").load(orders_silver_path)
df_orderlines = spark.read.format("delta").load(orderlines_silver_path)
df_customers = spark.read.format("delta").load(customers_silver_path)
df_weather = spark.read.format("delta").load(weather_silver_path)
df_holiday = spark.read.format("delta").load(holiday_silver_path)

print("ORDERS")
print(df_orders.columns)

print("\nORDERLINES")
print(df_orderlines.columns)

print("\nCUSTOMERS")
print(df_customers.columns)

print("\nWEATHER")
print(df_weather.columns)

print("\nHOLIDAY")
print(df_holiday.columns)

In [ ]:
print("Orders:", df_orders.count())
print("OrderLines:", df_orderlines.count())
print("Customers:", df_customers.count())
print("Weather:", df_weather.count())
print("Holiday:", df_holiday.count())

Confirm date coverage

In [ ]:
from pyspark.sql.functions import min, max, countDistinct

print("Orders date range:")
display(
    df_orders.select(
        min("OrderDate").alias("min_order_date"),
        max("OrderDate").alias("max_order_date"),
        countDistinct("OrderDate").alias("order_days")
    )
)

print("Weather date range:")
display(
    df_weather.select(
        min("weather_date").alias("min_weather_date"),
        max("weather_date").alias("max_weather_date")
    )
)

print("Holiday date range:")
display(
    df_holiday.select(
        min("holiday_date").alias("min_holiday_date"),
        max("holiday_date").alias("max_holiday_date")
    )
)

Build DimDate

In [ ]:
from pyspark.sql.functions import (
    sequence,
    to_date,
    explode,
    col,
    year,
    month,
    monthname,
    dayofmonth,
    dayofweek,
    dayofyear,
    weekofyear,
    quarter,
    date_format,
    when,
    lit
)

min_date = df_orders.select(
    to_date("OrderDate").alias("order_date")
).agg({"order_date": "min"}).first()[0]

max_date = df_orders.select(
    to_date("OrderDate").alias("order_date")
).agg({"order_date": "max"}).first()[0]

print("Date range:", min_date, "to", max_date)

df_dim_date = (
    spark.range(1)
    .select(
        explode(
            sequence(
                lit(min_date),
                lit(max_date)
            )
        ).alias("Date")
    )
    .select(
        col("Date"),
        date_format("Date", "yyyyMMdd").cast("int").alias("DateKey"),
        year("Date").alias("Year"),
        quarter("Date").alias("Quarter"),
        month("Date").alias("Month"),
        monthname("Date").alias("MonthName"),
        dayofmonth("Date").alias("Day"),
        dayofyear("Date").alias("DayOfYear"),
        dayofweek("Date").alias("DayOfWeek"),
        weekofyear("Date").alias("WeekOfYear"),
        date_format("Date", "EEEE").alias("DayName"),
        when(dayofweek("Date").isin([1, 7]), True)
            .otherwise(False)
            .alias("IsWeekend")
    )
)

print("DimDate rows:", df_dim_date.count())
display(df_dim_date.orderBy("Date").limit(20))

Write DimDate

In [ ]:
gold_date_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "gold/dim_date/"
)

(
    df_dim_date
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_date_path)
)

print("DimDate written successfully")

Verify

In [ ]:
df_dim_date_check = (
    spark.read
    .format("delta")
    .load(gold_date_path)
)

print("DimDate rows:", df_dim_date_check.count())

display(
    df_dim_date_check
    .orderBy("Date")
    .limit(10)
)

DimCustomer

In [ ]:
from pyspark.sql.functions import col

df_dim_customer = (
    df_customers
    .select(
        col("CustomerID").cast("int").alias("CustomerID"),
        col("CustomerName").cast("string").alias("CustomerName"),
        col("BillToCustomerID").cast("int").alias("BillToCustomerID"),
        col("CustomerCategoryID").cast("int").alias("CustomerCategoryID"),
        col("BuyingGroupID").cast("int").alias("BuyingGroupID"),
        col("DeliveryMethodID").cast("int").alias("DeliveryMethodID"),
        col("DeliveryCityID").cast("int").alias("DeliveryCityID"),
        col("PostalCityID").cast("int").alias("PostalCityID"),
        col("CreditLimit").cast("decimal(18,2)").alias("CreditLimit"),
        col("AccountOpenedDate").cast("date").alias("AccountOpenedDate"),
        col("StandardDiscountPercentage")
            .cast("decimal(5,2)")
            .alias("StandardDiscountPercentage"),
        col("IsStatementSent").cast("boolean").alias("IsStatementSent"),
        col("IsOnCreditHold").cast("boolean").alias("IsOnCreditHold"),
        col("PaymentDays").cast("int").alias("PaymentDays"),
        col("DeliveryRun").cast("string").alias("DeliveryRun"),
        col("RunPosition").cast("int").alias("RunPosition"),
        col("DeliveryPostalCode").cast("string").alias("DeliveryPostalCode"),
        col("PostalPostalCode").cast("string").alias("PostalPostalCode"),
        col("ValidFrom").cast("timestamp").alias("ValidFrom"),
        col("ValidTo").cast("timestamp").alias("ValidTo")
    )
)

print("DimCustomer rows:", df_dim_customer.count())
display(df_dim_customer.limit(10))

In [ ]:
print(
    "Null CustomerID:",
    df_dim_customer.filter(col("CustomerID").isNull()).count()
)

print(
    "Duplicate CustomerID:",
    df_dim_customer.groupBy("CustomerID")
    .count()
    .filter(col("count") > 1)
    .count()
)

Write DimCustomer to Gold

In [ ]:
gold_customer_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "gold/dim_customer/"
)

(
    df_dim_customer
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_customer_path)
)

print("DimCustomer written successfully")

Verify

In [ ]:
df_dim_customer_check = (
    spark.read
    .format("delta")
    .load(gold_customer_path)
)

print("DimCustomer rows:", df_dim_customer_check.count())
display(df_dim_customer_check.limit(10))

FactSales grain

We will use:

1 row = 1 OrderLine

So the expected fact count should be 231,412, matching your Silver OrderLines.

We'll join OrderLines → Orders using OrderID to obtain CustomerID and OrderDate.

In [ ]:
from pyspark.sql.functions import (
    col,
    to_date,
    date_format,
    round
)

df_fact_sales = (
    df_orderlines.alias("ol")
    .join(
        df_orders.alias("o"),
        col("ol.OrderID") == col("o.OrderID"),
        "inner"
    )
    .select(
        col("ol.OrderLineID").cast("long").alias("OrderLineID"),
        col("ol.OrderID").cast("long").alias("OrderID"),
        col("o.CustomerID").cast("int").alias("CustomerID"),

        to_date(col("o.OrderDate")).alias("OrderDate"),
        date_format(
            to_date(col("o.OrderDate")),
            "yyyyMMdd"
        ).cast("int").alias("OrderDateKey"),

        col("ol.StockItemID").cast("int").alias("StockItemID"),
        col("ol.PackageTypeID").cast("int").alias("PackageTypeID"),
        col("ol.Description").cast("string").alias("Description"),

        col("ol.Quantity").cast("int").alias("Quantity"),
        col("ol.UnitPrice").cast("decimal(18,2)").alias("UnitPrice"),
        col("ol.TaxRate").cast("decimal(5,2)").alias("TaxRate"),
        col("ol.PickedQuantity").cast("int").alias("PickedQuantity"),

        round(
            col("ol.Quantity") * col("ol.UnitPrice"),
            2
        ).cast("decimal(18,2)").alias("GrossSalesAmount"),

        round(
            col("ol.Quantity") *
            col("ol.UnitPrice") *
            col("ol.TaxRate") / 100,
            2
        ).cast("decimal(18,2)").alias("TaxAmount")
    )
)

print("FactSales rows:", df_fact_sales.count())

display(df_fact_sales.limit(10))


In [ ]:
print(
    "Duplicate OrderLineID:",
    df_fact_sales
    .groupBy("OrderLineID")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Null CustomerID:",
    df_fact_sales
    .filter(col("CustomerID").isNull())
    .count()
)

print(
    "Null OrderDateKey:",
    df_fact_sales
    .filter(col("OrderDateKey").isNull())
    .count()
)

print(
    "Invalid Quantity:",
    df_fact_sales
    .filter(col("Quantity") <= 0)
    .count()
)

print(
    "Invalid UnitPrice:",
    df_fact_sales
    .filter(col("UnitPrice") < 0)
    .count()
)

Write FactSales to Gold

In [ ]:
gold_fact_sales_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "gold/fact_sales/"
)

(
    df_fact_sales
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_fact_sales_path)
)

print("FactSales written successfully")

Verify

In [ ]:
df_fact_sales_check = (
    spark.read
    .format("delta")
    .load(gold_fact_sales_path)
)

print("FactSales rows:", df_fact_sales_check.count())

display(df_fact_sales_check.limit(10))

Not converting Weather and Holiday into FactSales, because your actual data periods don't overlap:
Orders:   2013-01-01 → 2016-05-31
Weather:  2025-01-01 → 2025-01-05
Holiday:  2025-01-01 → 2025-12-25

the daily sales aggregate

In [ ]:
from pyspark.sql.functions import (
    col,
    countDistinct,
    count,
    sum,
    avg,
    round
)

df_agg_daily_sales = (
    df_fact_sales
    .groupBy(
        "OrderDate",
        "OrderDateKey"
    )
    .agg(
        countDistinct("OrderID").alias("OrderCount"),
        count("OrderLineID").alias("OrderLineCount"),
        sum("Quantity").alias("TotalQuantity"),
        round(sum("GrossSalesAmount"), 2).alias("GrossSalesAmount"),
        round(sum("TaxAmount"), 2).alias("TaxAmount"),
        round(avg("GrossSalesAmount"), 2).alias("AverageOrderLineValue")
    )
    .orderBy("OrderDate")
)

display(df_agg_daily_sales)

In [ ]:
print("Rows:", df_agg_daily_sales.count())
print("Columns:", df_agg_daily_sales.columns)

Basic DQ

In [ ]:
print("Null OrderDate:", df_agg_daily_sales.filter(col("OrderDate").isNull()).count())
print("Null OrderDateKey:", df_agg_daily_sales.filter(col("OrderDateKey").isNull()).count())
print("Duplicate dates:",
      df_agg_daily_sales
      .groupBy("OrderDate")
      .count()
      .filter(col("count") > 1)
      .count())

Write to Gold

In [ ]:
gold_daily_sales_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "gold/agg_daily_sales/"
)

(
    df_agg_daily_sales
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_daily_sales_path)
)

Verify

In [ ]:
df_daily_sales_check = spark.read.format("delta").load(gold_daily_sales_path)

print("agg_daily_sales rows:", df_daily_sales_check.count())

display(
    df_daily_sales_check.orderBy("OrderDate").limit(10)
)